In [ ]:
! pip install -U langchain-text-splitters


You should consider upgrading via the 'F:\Programming\RAG_Project_1\venv_rag\Scripts\python.exe -m pip install --upgrade pip' command.


In [ ]:
! pip install -U langchain-groq

  Using cached distro-1.9.0-py3-none-any.whl (20 kB)
  Using cached sniffio-1.3.1-py3-none-any.whl (10 kB)


You should consider upgrading via the 'F:\Programming\RAG_Project_1\venv_rag\Scripts\python.exe -m pip install --upgrade pip' command.


In [28]:
! pip langchain-memory

ERROR: unknown command "langchain-memory"



In [3]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

In [4]:
# Step 1: Load PDF
loader = PyPDFLoader("Grandma's Bag of Stories by Sudha Murthy.pdf")
documents = loader.load()

print(f"Loaded {len(documents)} pages")



Loaded 136 pages


In [5]:
# Step 2: Split into chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

chunks = text_splitter.split_documents(documents)
print(f"Created {len(chunks)} chunks")


Created 466 chunks


In [6]:
# Step 3: Create embeddings
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1394.41it/s]


In [7]:
# Step 4: Store in FAISS
vectorstore = FAISS.from_documents(chunks, embeddings)

# Save locally
vectorstore.save_local("faiss_index")

print("FAISS index created and saved!")

FAISS index created and saved!


In [8]:
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings

# Step 1: Load embeddings
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# Step 2: Load FAISS index
vectorstore = FAISS.load_local("faiss_index", embeddings, allow_dangerous_deserialization=True)

# Step 3: Create retriever
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

# Step 4: Query
query = "What is this document about?"

docs = retriever.invoke(query)



Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5867.20it/s]


In [9]:
# Print retrieved chunks
for i, doc in enumerate(docs):
    print(f"\n--- Chunk {i+1} ---")
    print(doc.page_content)


--- Chunk 1 ---
being	imposed	on	the	subsequent	purchaser	and	without	limiting	the	rights	under	copyright	reserved
above,	no	part	of	this	publication	may	be	reproduced,	stored	in	or	introduced	into	a	retrieval	system,	or
transmitted	in	any	form	or	by	any	means	(electronic,	mechanical,	photocopying,	recording	or
otherwise),	without	the	prior	written	permission	of	both	the	copyright	owner	and	the	above-mentioned
publisher	of	this	book.

--- Chunk 2 ---
locales	is	entirely	coincidental.
ISBN	978-0-143-33362-3
This	digital	edition	published	in	2015.
e-ISBN:	978-8-184-75603-6
This	book	is	sold	subject	to	the	condition	that	it	shall	not,	by	way	of	trade	or	otherwise,	be	lent,	resold,
hired	out,	or	otherwise	circulated	without	the	publisher’s	prior	written	consent	in	any	form	of	binding	or
cover	other	than	that	in	which	it	is	published	and	without	a	similar	condition	including	this	condition

--- Chunk 3 ---
and	how	much	they	help	children	to	learn.	Hence	this	book.
I	hope,	with	these	storie

In [20]:
import os
import json
from langchain.chat_models import init_chat_model

# read the API key from the JSON file
with open("cred.json") as f:
    data = json.load(f)
    grok_api_key = data["grok_api_key"]

# Set environment variable
os.environ["GROQ_API_KEY"] = grok_api_key

def get_llm():
    """Initialize and return the LLM model."""
    return init_chat_model("llama-3.1-8b-instant", model_provider="groq")

In [ ]:
# Load Vector DB + Embeddings

from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings

def get_vectorstore():
    embeddings = HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2"
    )

    vectorstore = FAISS.load_local(
        "faiss_index",
        embeddings,
        allow_dangerous_deserialization=True
    )
    return vectorstore

In [18]:
# Function to create retriever from vectorstore

def get_retriever(vectorstore):
    return vectorstore.as_retriever(search_kwargs={"k": 3})

In [14]:
# Main RAG function
def rag_query(query):
    llm = get_llm()
    vectorstore = get_vectorstore()
    retriever = get_retriever(vectorstore)

    # Retrieve documents
    docs = retriever.invoke(query)

    context = "\n\n".join([doc.page_content for doc in docs])

    prompt = f"""
You are a helpful assistant.

Answer ONLY from the provided context.
If the answer is not in the context, say "I don't know".

Context:
{context}

Question:
{query}
"""

    response = llm.invoke(prompt)

    return response.content

In [26]:
query = "who is sudha murthy?"

answer = rag_query(query)

print("\nFinal Answer:\n", answer)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7030.21it/s]



Final Answer:
 Sudha Murty was a renowned writer in English and Kannada, and the chairperson of the Infosys Foundation.
